<a href="https://colab.research.google.com/github/mcjkurz/qhchina/blob/main/tutorials/Kunqu_GEN_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## English Kunqu Generation

In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
!wget https://raw.githubusercontent.com/mcjkurz/qhchina/refs/heads/main/tutorials/data/kunqu_dramas.txt

--2024-11-03 22:25:51--  https://raw.githubusercontent.com/mcjkurz/qhchina/refs/heads/main/tutorials/data/kunqu_dramas.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 365960 (357K) [text/plain]
Saving to: ‘kunqu_dramas.txt’

kunqu_dramas.txt    100%[===================>] 357.38K  --.-KB/s    in 0.007s  

2024-11-03 22:25:51 (47.5 MB/s) - ‘kunqu_dramas.txt’ saved [365960/365960]



In [5]:
# Load the text from kunqu_dramas.txt
with open("kunqu_dramas.txt", "r") as rf:
    kunqu_text = rf.read()

def preprocess(text):
    # Define the pattern to match only alphabetic characters, punctuation, new line symbols, and spaces
    pattern = r'[^a-zA-Z\s\n\.,!?;:\'\’\"\-() ]'

    # Use re.sub to replace all characters that do not match the pattern with an empty string
    cleaned_text = re.sub(pattern, '', text)

    lines = []
    for line in cleaned_text.split("\n"):
        line = line.strip()
        matches = re.search(r"[A-Z]{2,}", line)
        if len(line) > 4 and not matches:
            lines.append(line)

    cleaned_text = "\n".join(lines)

    return cleaned_text

kunqu_text = preprocess(kunqu_text)

kunqu_text[500:]

'aochang, the young lady\npreparing to become a nun. You know when we met,\nI could distinctly sense some tender feelings she felt\nfor me. See the quiet courtyard flooded with the\nintoxicating moonlight, I think ll go take a stroll\naround White Cloud Chamber, where Ms. Chen stays.\nPl do just that.\nPan sings:\nStroling along, I see the fallen petals on the path.\n(Exit Pan. Enter Chen Miaochang.)\nChen sings: See how reflections of the blossoms dance on the wall,\nAs bamboo trees and withered lotus sway in the autumn\nbreexe.\nOh, yes, to the bright moon will I play my xither,\nAss the gold incense burner revels in fragrance.\n(Recitative:)\nChen:\nMy name is Chen Miaochang. Lately ve been so\noccupied with all the mundane affairs in the nunnery\nthat I’ve quite neglected practicing my zither. It’s\nsuch a lovely, refreshing moon tonight. Let me play\nsomething dedicated to this sweet evening.\nChen sings: Indeed, I feel as if I were in the celestial palace.\n(Reenter Pan.)\nPan si

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [23]:
from transformers import pipeline

text_generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

# Example usage
prompt = "Wang and Zhou are going to the new canteen at the university.\nWang sings: How tasty is the new food here!"
generated_text = text_generator(
    prompt,
    max_length=200,  # Increased max_length for more varied outputs
    min_length=100,
    num_return_sequences=1,
    do_sample=True,
    top_k=30,
    num_beams=10,
    temperature=25.0,  # Adjusted temperature for more randomness
    top_p=0.95,  # Adjusted top-p for more diversity
    repetition_penalty=1.5
)
text = generated_text[0]["generated_text"]
lines = [line.strip() for line in re.split(r'(?<!\.)\.(?!\.)', text)]
text = ".\n".join(lines)
print(text)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Wang and Zhou are going to the new canteen at the university.
Wang sings: How tasty is the new food here! Chen’er is back from his illness, I’M not sure who will take me in now.
What would he think of me like that? But what would Zhu Shi do after reading this report? Who would dare punish a human spirit with such force, if that did not make him laugh! This kind of speech does not deserve my heart! Zhihong : No one here, so why not you?


In [15]:
from datasets import Dataset

# Tokenize the text
tokens = tokenizer(kunqu_text)["input_ids"]

# Split tokenized text into overlapping chunks while ensuring words are not split
chunk_size = 256
stride = chunk_size // 3
chunks = []

i = 0
while i < len(tokens):
    end = i + chunk_size
    chunk = tokens[i:end]

    # Adjust the chunk to avoid splitting subwords
    if end < len(tokens):
        # Move the end pointer back to avoid splitting a word
        while end < len(tokens) and tokenizer.convert_ids_to_tokens(tokens[end]).startswith('##'):
            end -= 1
        chunk = tokens[i:end]

    chunks.append(chunk)
    i += stride

# Convert chunks back to string format for dataset creation
text_chunks = [" ".join(tokenizer.decode(torch.tensor(chunk), skip_special_tokens=True).split()) for chunk in chunks]

# Create a Dataset
dataset = Dataset.from_dict({"text": text_chunks})

print(dataset)
print(dataset[5])

Token indices sequence length is longer than the specified maximum sequence length for this model (71149 > 1024). Running this sequence through the model will result in indexing errors


Dataset({
    features: ['text'],
    num_rows: 838
})
{'text': "The Jade Pin Wait! Is it the wind I’m hearing or...how intriguing! A xither, yes, the sound intoxicating. Who's playing so adroitly - who can it be? Is it played to the moon, or to melancholy? Chen sings: Alas, my sadness will not be dispelled with the music. (Recitative by Pan: The door is ajar. Ah, it’s the maiden nun-to-be Chen Miaochang, playing the zither. Let me enter.) All I can do is to sigh to the autumn wind. (Recitative by Pan: Brava, brava!) How tis it that you, young master Pan, are here in my quarters? You have given me quite a start! (Recitative:) Chen: Oh, please forgive me for startling you, Miss. I was just strolling about. Young master Pan: Chen sings: So this cool autumn night's music has brought you here. (Recitative:) Indeed. I was drawn by the exquisite songs and your clear, beautiful voice accompanied by the zither. Chen"}


In [16]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AdamW, get_scheduler
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=chunk_size)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask'])

train_dataloader = DataLoader(tokenized_datasets, shuffle=True, batch_size=4)

optimizer = AdamW(model.parameters(), lr=1e-5)
num_epochs = 3
num_training_steps = len(train_dataloader) * num_epochs

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Training loop
model.train()
progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(input_ids = batch['input_ids'], attention_mask = batch['attention_mask'], labels=batch['input_ids'])
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        progress_bar.set_description(f"Epoch {epoch+1}")
        progress_bar.set_postfix(loss=loss.item())
        progress_bar.update(1)

Map:   0%|          | 0/838 [00:00<?, ? examples/s]

  0%|          | 0/630 [00:00<?, ?it/s]

In [18]:
from transformers import pipeline

# Example usage
prompt = "Wang and Zhou are going to the new canteen at the university.\nWang sings: How tasty is the new food here!"
generated_text = text_generator(
    prompt,
    max_length=200,  # Increased max_length for more varied outputs
    min_length=100,
    num_return_sequences=1,
    do_sample=True,
    top_k=30,
    num_beams=10,
    temperature=25.0,  # Adjusted temperature for more randomness
    top_p=0.95,  # Adjusted top-p for more diversity
    repetition_penalty=1.5
)

text = generated_text[0]["generated_text"]
lines = [line.strip() for line in re.split(r'(?<!\.)\.(?!\.)', text)]
text = ".\n".join(lines)
print(text)

Wang and Zhou are going to the new canteen at the university.
Wang sings: How tasty is the new food here! What can I ever be of rice even though all this good people in the north have arrived! All these old masters who taught me about medicine are returning, with a view to our coming reunion, my dear.
Pll leave now.
Why not take him back? He'll never return to his aunt if we keep doing this thing together - how disgusting! I know he shouldn


# Chinese Kunqu Generation

In [1]:
!pip install pycantonese
!pip install opencc-python-reimplemented
!pip install pypinyin
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 834.7/834.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beh

In [2]:
!wget https://raw.githubusercontent.com/mcjkurz/qhchina/refs/heads/main/tutorials/data/kunqu_dramas.txt

--2024-11-03 22:43:59--  https://raw.githubusercontent.com/mcjkurz/qhchina/refs/heads/main/tutorials/data/kunqu_dramas.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 365960 (357K) [text/plain]
Saving to: ‘kunqu_dramas.txt’

kunqu_dramas.txt    100%[===================>] 357.38K  --.-KB/s    in 0.03s   

2024-11-03 22:43:59 (12.7 MB/s) - ‘kunqu_dramas.txt’ saved [365960/365960]



In [3]:
from pypinyin import pinyin, Style
import opencc
import pycantonese

# Initialize the converter for Simplified to Traditional
converter = opencc.OpenCC('s2t')

# Define the function to classify 平仄 based on Mandarin and Cantonese
def classify_ping_ze(character, exceptions=None):

    if exceptions and character in exceptions:
        return exceptions[character]

    mandarin_pinyin = pinyin(character, style=Style.TONE3, heteronym=False)[-1]
    traditional_char = converter.convert(character)
    jyutping_list = pycantonese.characters_to_jyutping(traditional_char)

    if not jyutping_list or jyutping_list[0][1] is None:
        return None  # No Jyutping data found
    jyutping = jyutping_list[0][1]  # Get the Jyutping string
    if len(jyutping) > 1 and jyutping[-2] in ('p', 't', 'k'):
        return "仄"  # 入声 is classified as 仄

    if not mandarin_pinyin:
        return None  # No tone information available, return None

    tone_number = int(mandarin_pinyin[-1][-1]) if mandarin_pinyin[-1][-1] in '1234' else None

    if tone_number in [1, 2]:
        return "平"  # 1st or 2nd tone is 平
    elif tone_number in [3, 4]:
        return "仄"  # 3rd tone is 仄

    return None  # If no classification was made, return None

In [6]:
pycantonese.characters_to_jyutping("食")

[('食', 'sik6')]

In [7]:
classify_ping_ze('狹')

'仄'

In [29]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model and tokenizer
model_name = "uer/gpt2-chinese-cluecorpussmall"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
# Step 1: Create a vocabulary map for 平 and 仄
vocab = tokenizer.get_vocab()
ping_chars = set()
ze_chars = set()
none_chars = set()

for char in vocab.keys():
    if not char.startswith("##"):
        category = classify_ping_ze(char)
    else:
        category = classify_ping_ze(char[2:])

    if category == "平":
        ping_chars.add(tokenizer.convert_tokens_to_ids(char))
    elif category == "仄":
        ze_chars.add(tokenizer.convert_tokens_to_ids(char))
    else:
        none_chars.add(tokenizer.convert_tokens_to_ids(char))

In [10]:
def find_vowel(jyutping):
    # Define possible vowels and diphthongs
    vowels = ['aa', 'a', 'e', 'i', 'o', 'u', 'oe', 'yu']

    # Start from the end of the string and find the last occurrence of a vowel
    for i in range(len(jyutping) - 1, -1, -1):
        for vowel in vowels:
            if jyutping[i - len(vowel) + 1:i + 1] == vowel:
                # Return the rhyme, which is the vowel and everything following it
                return jyutping[i - len(vowel) + 1:]

    return None  # No vowel found, meaning it doesn't have a recognizable rhyme

def get_rhyming_ids(rhyming_char, vocab, tokenizer):
    # Get the Jyutping for the rhyming character
    rhyming_char = converter.convert(rhyming_char)
    jyutping_list = pycantonese.characters_to_jyutping(rhyming_char)

    if not jyutping_list:
        return set()  # No Jyutping data, return an empty set

    # Determine the rhyme from the Jyutping of the rhyming character
    rhyme = find_vowel(jyutping_list[0][1])

    if rhyme is None:
        return set()  # No recognizable rhyme, return an empty set

    # Initialize a set to store rhyming token IDs
    rhyming_ids = set()

    # Go through the vocabulary and find tokens that match the rhyme
    for char in vocab.keys():
        if char.startswith("##"):
            jyutping_word = char[2:]
        else:
            jyutping_word = char

        # Get Jyutping and check if it matches the rhyme
        jyutping_list = pycantonese.characters_to_jyutping(jyutping_word)
        #print(jyutping_word, jyutping_list)

        if jyutping_list and jyutping_list[0][1] is not None:
            jyutping_rhyme = find_vowel(jyutping_list[0][1])
            if jyutping_rhyme == rhyme:
                rhyming_ids.add(tokenizer.convert_tokens_to_ids(char))

    return rhyming_ids

# Step 2: Define the function to generate text based on 平仄 pattern
def generate_with_ping_ze(pattern, existing_text="", no_generate=["他","她","的"]):
    model.eval()

    # Initialize input_ids based on existing text (if any)
    input_ids = torch.tensor([tokenizer.cls_token_id]).unsqueeze(0).to(device)
    generated_text = []

    # Find the last 平 positions in each line for rhyming
    line_end_indices = [i for i, char in enumerate(pattern) if char in {'。'}]
    rhyme_positions = [i - 1 for i in line_end_indices]
    rhyming_word = None

    for i, tone in enumerate(pattern):
        # If the current position has a punctuation in the pattern, add it directly
        if tone in {'，', '。', ',', '.', '、', ' '}:  # Add any other symbols as needed
            token_id = tokenizer.convert_tokens_to_ids(tone)
            input_ids = torch.cat((input_ids, torch.tensor([[token_id]]).to(device)), dim=1)
            generated_text.append(tone)
            continue

        # Check if we have an existing character at this position
        if i < len(existing_text) and existing_text[i] != 'X':
            # Use the existing character
            char = existing_text[i]
            generated_text.append(char)
            token_id = tokenizer.convert_tokens_to_ids(char)
            input_ids = torch.cat((input_ids, torch.tensor([[token_id]]).to(device)), dim=1)
            if i in rhyme_positions and not rhyming_word:
                rhyming_word = char  # The last character generated for rhyme
            continue

        # Generate a new character based on the 平 or 仄 requirement
        outputs = model(input_ids)
        logits = outputs.logits[:, -1, :].clone()

        # Mask logits according to 平 or 仄 requirement
        if tone == "平":
            logits[0, list(ze_chars)] = -float("inf")
        elif tone == "仄":
            logits[0, list(ping_chars)] = -float("inf")

        # Mask out characters that aren't classified as 平 or 仄
        logits[0, list(none_chars)] = -float("inf")

        if no_generate:
          no_generate_ids = tokenizer.convert_tokens_to_ids(no_generate)
          logits[0, no_generate_ids] = -float("inf")

        # Apply rhyme constraints if we're at a rhyming position
        if i in rhyme_positions:
            if rhyming_word:
                # Use the last generated character from the first line as the rhyming word
                rhyme_chars = get_rhyming_ids(rhyming_word, vocab, tokenizer)
                # Mask out all characters not in the rhyme group
                logits[0, list(set(range(logits.size(-1))) - rhyme_chars)] = -float("inf")


        for char in generated_text:
            char_id = tokenizer.convert_tokens_to_ids(char)
            logits[0, char_id] = -float("inf")

        if torch.isinf(logits).all():
            # If all logits are -inf, relax the rhyme constraint entirely
            print("No valid rhyming characters available; relaxing rhyme constraint.")
            # Reset logits to allow any character with the specified tone
            logits = outputs.logits[:, -1, :].clone()  # Reset to original logits
            if tone == "平":
                logits[0, list(ze_chars)] = -float("inf")  # Allow any ping_chars
            elif tone == "仄":
                logits[0, list(ping_chars)] = -float("inf")  # Allow any ze_chars

        # Sample from the probability distribution
        probs = torch.softmax(logits, dim=-1)
        next_token_id = torch.multinomial(probs, num_samples=1)[-1]

        # Decode and add generated character to the text
        next_char = tokenizer.decode(next_token_id)
        if next_char.startswith("##"):
          next_char = next_char[2:]
        if i in rhyme_positions and not rhyming_word:
            rhyming_word = next_char  # The last character generated for rhyme
        generated_text.append(next_char)
        input_ids = torch.cat((input_ids, next_token_id.unsqueeze(0)), dim=1).to(device)

    print(pattern)
    return "".join(generated_text)

In [11]:
# Example usage
pattern = "平平仄仄平平仄，仄仄平平仄仄平。仄仄平平平仄仄，平平仄仄仄平平。"
result = generate_with_ping_ze(pattern)
print(result)

平平仄仄平平仄，仄仄平平仄仄平。仄仄平平平仄仄，平平仄仄仄平平。
冬天去吃双拼是，量少而腥气十分。服务员来加水要，之前问下价钱真。


https://www.52shici.com/zd/shi.php

https://www.duiduilian.com/pzcx/

https://www.poemshenzhen.com/index.php/Testing/index

In [30]:
from transformers import GPT2Tokenizer
import torch
import re

# Load the text from kunqu_dramas.txt
with open("kunqu_dramas.txt", "r") as rf:
    kunqu_text = rf.read()

def preprocess(text):
    # Define the pattern to match only alphabetic characters, punctuation, new line symbols, and spaces
    pattern = r'[a-zA-Z0-9\s\n\.>&</,!?;:\'\’\"\-()$#®\|\“\”\[\]\*\°\\\=\_£©《》「」{} ]'
    titles = r"玉答记|烂柯山|狮吼记|昭君出塞|孽海记|虎囊弹|牡丹亭|钟馗嫁妹|长生殿"

    # Use re.sub to replace all characters that do not match the pattern with an empty string
    cleaned_text = re.sub(pattern, '', text)
    cleaned_text = re.sub(titles, '', cleaned_text)

    lines = []
    for line in cleaned_text.split("\n"):
        line = line.strip()
        matches = re.search(r"[A-Z]{2,}", line)
        if len(line) > 4 and not matches:
            lines.append(line)

    cleaned_text = "\n".join(lines)

    return cleaned_text

kunqu_text = preprocess(kunqu_text)
kunqu_text[500:]

'。情儿意儿，哪些儿不动人她独自理瑶琴，我独立区苔冷。分明是西厢行径。老天哪早早成就少年秦晋，少年秦晋。姑母上唱陈师父。姑母罢了。我侄儿染病书斋，我去看看就陈师父，既是潘相公有病，徒弟陪师父—姑母使得。促子陈—人间无羞归因下第，药治相竟害至相思病，服药无效。格末那处。让我扶促出来，散散心，或许好§阿要到外面来坐坐吧使得。风雨莫秋天气。客窗风雨恨长天。咳，许多心事向谁言进安病里偏教人易老，秋来转觉恨常添。只因心事有相牵。进安。相公。我自别家乡，旅馆凄凉，不觉染成一病，如何是好相公，弓番道革，我前日蔡相公起了课土怎么说说相公格病是阴人面上来格，忽消吃些许进安小事，就该与我送了。得脱介潘进安姑母转过荷香径，进安%使我更关心。格个宝贝倒也来格。啊，侄儿，这两日病体如何了小道在此。在郧里看仔细等好子勒谢，来得及革。侄儿，你把起病根由说与我知道。—§进安潘唱这病儿何曾经害这病儿好难担符。这病儿好似风前败叶，姑母你有甚么心事，说与我知道。风寒上起的进安也纪是格。潘也不是。进安莫不是立西风病儿已深。莫不是费心情勤读再。进安我好钝啊泡杯茶休吃吧不消。这倒使得。月圆月缺，月也有盈亏害，襟怀，你把那段心儿且放开。啊，潘相公，还是服药为上。咳，吃什么药，心病还需心药医。妖怪速退。好笑我但相公，直格病重，听见陈姑来看但，直立个立起来，说道有劳我方才送姑奶奶出去，陈姑对我说道进安，你家相公的病我都已知道了。啊，她竟晓得了。晓得个哉。说我有绝妙的方儿，叫你家相公到我房中来，写上一帖，他的病就好了。方儿要你到她房中去写。如此，待我就去。格副样式，那好去介待我除下帕儿。解脱子包头布。相公到落里去到陈姑房中写方儿去。哈人说的是你方才说的是格条裙，自家带子进去。只此一遭，下不为个。相会，你这病是装出来格是装出来的讨西风别院，黄菊都开遍。秋燕不知人意懒，对对飞来池畔。想我在此出家，只为身无所归，潘郎之后，好生伤感。原非本心，寄迹于此。那知弄假成真，到后来，不知怎生结果。有纸笔在此，不免把心事写作一词，聊寄幽情，消遗闷怀。松舍青灯闪闪，黄昏独自展孤会，—万般情意难禁，强将经卷压凡心，沉沉病染相全恨无眠残月窗西。——堆积，我几番长叹空自翡。想我在此出家，倘然不了，怎么处理怕春去留不住少年颜色。云房静，欲求仙，恨着天人台路迷。小生病起无聊，闲游潜行，不党已到白云楼下。好一阵扑鼻清香也。问津何处，傍青松，掩映着花

In [27]:
from datasets import Dataset

# Tokenize the text
tokens = tokenizer(kunqu_text)["input_ids"]

# Split tokenized text into overlapping chunks while ensuring words are not split
chunk_size = 36
stride = chunk_size // 3
chunks = []

i = 0
while i < len(tokens):
    end = i + chunk_size
    chunk = tokens[i:end]

    # Adjust the chunk to avoid splitting subwords
    if end < len(tokens):
        # Move the end pointer back to avoid splitting a word
        while end < len(tokens) and tokenizer.convert_ids_to_tokens(tokens[end]).startswith('##'):
            end -= 1
        chunk = tokens[i:end]

    chunks.append(chunk)
    i += stride

# Convert chunks back to string format for dataset creation
text_chunks = ["".join(tokenizer.decode(torch.tensor(chunk), skip_special_tokens=True).split()) for chunk in chunks]

# Create a Dataset
dataset = Dataset.from_dict({"text": text_chunks})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 1854
})


In [31]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AdamW, get_scheduler
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=chunk_size)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# Create a DataLoader
train_dataloader = DataLoader(tokenized_datasets, shuffle=True, batch_size=4)

# Set up the optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=1e-5)
num_epochs = 5
num_training_steps = len(train_dataloader) * num_epochs

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Training loop
model.train()
progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(input_ids = batch['input_ids'], attention_mask = batch['attention_mask'], labels=batch['input_ids'])
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        progress_bar.set_description(f"Epoch {epoch+1}")
        progress_bar.set_postfix(loss=loss.item())
        progress_bar.update(1)

Map:   0%|          | 0/1854 [00:00<?, ? examples/s]

  0%|          | 0/2320 [00:00<?, ?it/s]

In [32]:
pattern = "平平仄仄平平仄，仄仄平平仄仄平。仄仄平平平仄仄，平平仄仄仄平平。"
result = generate_with_ping_ze(pattern)
#result = generate_with_ping_ze(pattern, existing_text="谁家月夜琴三弄，")
print(result)

平平仄仄平平仄，仄仄平平仄仄平。仄仄平平平仄仄，平平仄仄仄平平。
曾前往列宾台就，不是随邦用薯兵。体淡风轻珠泪未，生平只见枉容兄。


In [3]:
from transformers import pipeline
import re

text_generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

# Example usage
prompt = "多么好的地方！"
generated_text = text_generator(
    prompt,
    max_length=200,  # Increased max_length for more varied outputs
    min_length=100,
    num_return_sequences=1,
    do_sample=True,
    top_k=30,
    num_beams=10,
    temperature=25.0,  # Adjusted temperature for more randomness
    top_p=0.95,  # Adjusted top-p for more diversity
    repetition_penalty=1.5
)

text = "".join(generated_text[0]["generated_text"].split())
lines = [line.strip() for line in re.split(r'。', text)]
text = ".\n".join(lines)
print(text)

多么好的地方！何娘子好像要去.
不妨，今朝被人杀了，也算是我的错，怪我你圣没文原由请君引杜柳这就使只.青天既降此下大'真良武目许和别有全主镜猜生綱過顧導從麗非瘋侶競爾啊塵蒼相烏>許機如蘇上備叢-荊櫻濱


# Loading Fine-Tuned Models (Back-Up)


In [22]:
# English

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model and tokenizer
model_name = "qhchina/gpt2-kunqu-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [1]:
# Chinese

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model and tokenizer
model_name = "qhchina/gpt2-kunqu-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/408M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]